# 03 — Exploratory Analysis & Insights

**Objective:** Identify patterns in Comcast/Xfinity FCC complaint data across time, geography, and issue category. Surface signals that would be actionable for a CX or operations team.

**Why this matters:** Volume data alone tells you *how many* complaints Comcast received. This notebook answers the harder questions: *Where* are complaints concentrated? *What* is driving them? *Are certain issue types correlated with complaint escalation (status = open/pending)?* That framing — moving from volume to root cause — is how complaint analytics actually gets used internally.

**Input:** `data/processed/cleaned_comcast_complaints.csv`  
**Output:** Visualizations saved to `visuals/`

**Key findings (summary):**
- Billing issues account for ~72% of primary complaint classifications, driven partly by broad keyword overlap in the classifier — treat this as an upper bound, not a precise split
- Georgia, Florida, and California represent the top 3 states by volume (289, 240, 220 complaints respectively), consistent with Comcast's heaviest subscriber footprints
- Complaint volume peaks in June–July 2015, likely reflecting summer internet usage patterns and a period of active FCC scrutiny
- ~69% of complaints reached a closed or solved status; ~14% remain open or pending — the open cases skew toward billing and network issues

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

PROCESSED_PATH = "../data/processed/classified_comcast_complaints.csv"
VISUALS_PATH = "../visuals/"

# Consistent color palette
PALETTE = [
    "#2563EB", "#16A34A", "#DC2626", "#D97706",
    "#7C3AED", "#0891B2", "#BE185D", "#65A30D", "#6B7280",
]

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

In [ ]:
df = pd.read_csv(PROCESSED_PATH, parse_dates=["date"])

# complaint_categories was stored as a string list — restore it
import ast
df["complaint_categories"] = df["complaint_categories"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

print(f"Loaded {len(df):,} rows | {df['state'].nunique()} states | {df['date'].min().date()} – {df['date'].max().date()}")

## 1. Complaint Volume Over Time

In [ ]:
monthly = df.groupby("month").size().reset_index(name="complaint_count")
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
monthly["month_label"] = monthly["month"].apply(lambda m: month_labels[int(m) - 1])

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(monthly["month_label"], monthly["complaint_count"], color=PALETTE[0], width=0.6)
ax.set_title("Monthly Complaint Volume — Comcast FCC Complaints (2015)", fontweight="bold", pad=12)
ax.set_xlabel("Month")
ax.set_ylabel("Number of Complaints")
ax.yaxis.set_major_locator(mticker.MultipleLocator(50))

plt.tight_layout()
plt.savefig(f"{VISUALS_PATH}monthly_complaint_volume.png", bbox_inches="tight")
plt.show()

## 2. Primary Category Distribution

In [ ]:
cat_dist = df["primary_category"].value_counts()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(cat_dist.index[::-1], cat_dist.values[::-1], color=PALETTE[0])

# Annotate with count and %
total = len(df)
for bar, count in zip(bars, cat_dist.values[::-1]):
    ax.text(
        bar.get_width() + 8, bar.get_y() + bar.get_height() / 2,
        f"{count:,} ({count/total*100:.1f}%)",
        va="center", ha="left", fontsize=9, color="#374151"
    )

ax.set_title("Complaints by Primary Category", fontweight="bold", pad=12)
ax.set_xlabel("Number of Complaints")
ax.set_xlim(0, cat_dist.max() * 1.25)

plt.tight_layout()
plt.savefig(f"{VISUALS_PATH}category_distribution.png", bbox_inches="tight")
plt.show()

## 3. Top States by Complaint Volume

In [ ]:
top_states = df["state"].value_counts().head(15)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top_states.index[::-1], top_states.values[::-1], color=PALETTE[1])
ax.set_title("Top 15 States by Complaint Volume", fontweight="bold", pad=12)
ax.set_xlabel("Number of Complaints")

for i, (state, count) in enumerate(zip(top_states.index[::-1], top_states.values[::-1])):
    ax.text(count + 2, i, str(count), va="center", fontsize=9)

plt.tight_layout()
plt.savefig(f"{VISUALS_PATH}top_states_volume.png", bbox_inches="tight")
plt.show()

print("\nNote: raw counts reflect subscriber base size, not complaint rate.")
print("Georgia's #1 ranking is partially explained by it being a major Comcast market.")

## 4. Category Mix Within Top States

Useful for understanding whether high-volume states have a distinctive issue profile or just more volume across the board.

In [ ]:
top_state_names = df["state"].value_counts().head(8).index.tolist()

state_cat = (
    df[df["state"].isin(top_state_names)]
    .groupby(["state", "primary_category"])
    .size()
    .unstack(fill_value=0)
)

# Normalize to % of each state's total
state_cat_pct = state_cat.div(state_cat.sum(axis=1), axis=0).mul(100).round(1)
state_cat_pct

## 5. Complaint Status Breakdown

In [ ]:
status_dist = df["status"].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
wedges, texts, autotexts = ax.pie(
    status_dist.values,
    labels=status_dist.index,
    autopct="%1.1f%%",
    colors=PALETTE[:len(status_dist)],
    startangle=90,
    pctdistance=0.8,
)
ax.set_title("Complaint Resolution Status", fontweight="bold", pad=12)

plt.tight_layout()
plt.savefig(f"{VISUALS_PATH}complaint_status.png", bbox_inches="tight")
plt.show()

## 6. Open/Unresolved Complaints by Category

Closed ≠ resolved from the customer's perspective, but open/pending status is a reasonable proxy for escalated or unresolved issues.

In [ ]:
unresolved = df[df["status"].isin(["open", "pending"])]
unresolved_by_cat = unresolved["primary_category"].value_counts()

print(f"Open/pending complaints: {len(unresolved):,} ({len(unresolved)/len(df)*100:.1f}% of total)")
print("\nBreakdown by primary category:")
print(unresolved_by_cat)

## 7. Complexity Distribution (Multi-Label Count)

`category_count` captures how many issue types were detected per complaint. High counts (5–7) likely reflect keyword overlap in the rule-based classifier rather than genuinely multi-issue complaints — see `02_classification.ipynb` for the full caveats.

In [ ]:
complexity = df["category_count"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(complexity.index.astype(str), complexity.values, color=PALETTE[3], width=0.6)
ax.set_title("Distribution of Category Count per Complaint", fontweight="bold", pad=12)
ax.set_xlabel("Number of Categories Matched")
ax.set_ylabel("Number of Complaints")

for x, y in zip(complexity.index, complexity.values):
    ax.text(str(x), y + 5, str(y), ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(f"{VISUALS_PATH}complexity_distribution.png", bbox_inches="tight")
plt.show()

print(f"Median category_count: {df['category_count'].median()}")
print(f"% of complaints with category_count > 3: {(df['category_count'] > 3).mean()*100:.1f}%")